In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
!pip install -q dagshub mlflow

import dagshub
import mlflow
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
DAGSHUB_TOKEN = user_secrets.get_secret("DAGSHUB_TOKEN")
DAGSHUB_USERNAME = user_secrets.get_secret("DAGSHUB_USERNAME")

repo_name = "ml_ass1"

mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USERNAME}/{repo_name}.mlflow")

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN

dagshub.init(repo_name=repo_name, repo_owner=DAGSHUB_USERNAME, mlflow=True)

print("Connected to DagsHub + MLflow successfully!")

Accessing as akeke23

Initialized MLflow to track repo "akeke23/ml_ass1"

Repository akeke23/ml_ass1 initialized!

Connected to DagsHub + MLflow successfully!


In [3]:
import pandas as pd
import numpy as np
import joblib


test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")
submission_ids = test["Id"]


import mlflow.sklearn
import mlflow.artifacts

# run_id = "9506993990d54e32b05a69ec2edbbf07" 

model_uri = "models:/HousePriceChampion/latest"
model = mlflow.sklearn.load_model(model_uri)
print("Model loaded from Registry!")

client = mlflow.tracking.MlflowClient()
version = client.get_latest_versions("HousePriceChampion", stages=["None"])[0]
run_id = version.run_id

local_dir = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="model", dst_path=".")

scaler         = joblib.load(os.path.join(local_dir, "scaler.pkl"))
ohe            = joblib.load(os.path.join(local_dir, "ohe.pkl"))
final_features = joblib.load(os.path.join(local_dir, "final_features.pkl"))


print(f"Artifacts loaded from Run: {run_id}")



cols_to_drop = [
    "Id","PoolQC","MiscFeature","Alley","Utilities","Street",
    "PoolArea","Condition2","RoofMatl","3SsnPorch",
    "LowQualFinSF","Heating","MiscVal","KitchenAbvGr"
]
test = test.drop(columns=[c for c in cols_to_drop if c in test.columns])

def impute(df):
    num_medians = {'MSSubClass': 50.0, 'LotFrontage': 69.0, 'LotArea': 9478.5, 'OverallQual': 6.0, 
                   'OverallCond': 5.0, 'YearBuilt': 1973.0, 'YearRemodAdd': 1994.0, 'MasVnrArea': 0.0, 
                   'BsmtFinSF1': 383.5, 'BsmtFinSF2': 0.0, 'BsmtUnfSF': 477.5, 'TotalBsmtSF': 991.5, 
                   '1stFlrSF': 1087.0, '2ndFlrSF': 0.0, 'GrLivArea': 1464.0, 'BsmtFullBath': 0.0, 
                   'BsmtHalfBath': 0.0, 'FullBath': 2.0, 'HalfBath': 0.0, 'BedroomAbvGr': 3.0, 
                   'KitchenAbvGr': 1.0, 'TotRmsAbvGrd': 6.0, 'Fireplaces': 1.0, 'GarageYrBlt': 1980.0, 
                   'GarageCars': 2.0, 'GarageArea': 480.0, 'WoodDeckSF': 0.0, 'OpenPorchSF': 25.0, 
                   'EnclosedPorch': 0.0, 'ScreenPorch': 0.0, 'MoSold': 6.0, 'YrSold': 2008.0}
    
    cat_modes = {'MSZoning': 'RL', 'LotShape': 'Reg', 'LandContour': 'Lvl', 'LotConfig': 'Inside', 
                 'LandSlope': 'Gtl', 'Neighborhood': 'NAmes', 'Condition1': 'Norm', 'BldgType': '1Fam', 
                 'HouseStyle': '1Story', 'RoofStyle': 'Gable', 'Exterior1st': 'VinylSd', 
                 'Exterior2nd': 'VinylSd', 'MasVnrType': 'BrkFace', 'ExterQual': 'TA', 'ExterCond': 'TA', 
                 'Foundation': 'PConc', 'BsmtQual': 'TA', 'BsmtCond': 'TA', 'BsmtExposure': 'No', 
                 'BsmtFinType1': 'Unf', 'BsmtFinType2': 'Unf', 'HeatingQC': 'Ex', 'CentralAir': 'Y', 
                 'Electrical': 'SBrkr', 'KitchenQual': 'TA', 'Functional': 'Typ', 'FireplaceQu': 'Gd', 
                 'GarageType': 'Attchd', 'GarageFinish': 'Unf', 'GarageQual': 'TA', 'GarageCond': 'TA', 
                 'PavedDrive': 'Y', 'Fence': 'MnPrv', 'SaleType': 'WD', 'SaleCondition': 'Normal'}

    for col, val in num_medians.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)

    for col, val in cat_modes.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)
    return df

test = impute(test)

def add_features(df):
    df = df.copy()
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["TotalBathrooms"] = df["FullBath"] + 0.5*df["HalfBath"] + df["BsmtFullBath"] + 0.5*df["BsmtHalfBath"]
    df["TotalPorchSF"] = df["OpenPorchSF"] + df["EnclosedPorch"] + df["ScreenPorch"]
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["GarageAge"] = df["YrSold"] - df["GarageYrBlt"]
    df["Qual_x_GrLivArea"] = df["OverallQual"] * df["GrLivArea"]
    df["TotalBsmt"] = df["BsmtFinSF1"] + df["BsmtFinSF2"] + df["BsmtUnfSF"]
    df["RoomsPerArea"] = df["TotRmsAbvGrd"] / (df["GrLivArea"] + 1)
    return df

test = add_features(test)

drop2 = [
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF", 
    "1stFlrSF", "2ndFlrSF", "FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath",
    "YearBuilt", "YearRemodAdd", "GarageYrBlt", "YrSold",
    "OpenPorchSF", "EnclosedPorch", "ScreenPorch", "TotRmsAbvGrd"
]
test = test.drop(columns=[c for c in drop2 if c in test.columns])


skewed_features = [
    'LotArea','LotFrontage','MasVnrArea','TotalPorchSF',
    'Qual_x_GrLivArea','TotalSF','TotalBsmt',
    'WoodDeckSF','MSSubClass','GrLivArea'
]
for col in skewed_features:
    if col in test.columns:
        test[col] = np.log1p(test[col])

cat_cols = test.select_dtypes(include='object').columns
num_cols = test.select_dtypes(exclude='object').columns

encoded = ohe.transform(test[cat_cols])

encoded_df = pd.DataFrame(
    encoded,
    columns=ohe.get_feature_names_out(cat_cols),
    index=test.index
)

X_test = pd.concat([test[num_cols], encoded_df], axis=1)


X_test = X_test.reindex(columns=scaler.feature_names_in_, fill_value=0)
X_test_scaled_full = scaler.transform(X_test)

X_test_scaled_df = pd.DataFrame(X_test_scaled_full, columns=scaler.feature_names_in_)

X_test_final = X_test_scaled_df[final_features]


preds_log = model.predict(X_test_final)
preds = np.expm1(preds_log)

submission = pd.DataFrame({
    "Id": submission_ids,
    "SalePrice": preds
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Done! Submission saved.")
print(submission.head())

Model loaded from Registry!


/tmp/ipykernel_127/3910356091.py:20: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  version = client.get_latest_versions("HousePriceChampion", stages=["None"])[0]


Artifacts loaded from Run: 9506993990d54e32b05a69ec2edbbf07
Done! Submission saved.
     Id      SalePrice
0  1461  113064.444520
1  1462  154938.156636
2  1463  181097.336712
3  1464  201663.291790
4  1465  208032.610853
